# Unique OpWLS Photon Counts

This notebook reads the `tools::wcsv::ntuple` CSV files in `build/output`, filters rows with `ProcessName == "OpWLS"`, and counts unique photons by `(EventID, TrackID)`.

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("../build/output")
CSV_GLOB = "*.csv"
UNIQUE_KEYS = ["EventID", "TrackID"]
PROCESS_NAME = "OpWLS"

In [ ]:
def read_tools_wcsv(path: Path) -> pd.DataFrame:
    columns = []
    with path.open() as handle:
        for line in handle:
            if line.startswith("#column "):
                columns.append(line.strip().split()[-1])
            elif not line.startswith("#"):
                break

    if not columns:
        raise ValueError(f"No #column metadata found in {path}")

    df = pd.read_csv(path, comment="#", names=columns)
    df["SourceFile"] = path.name
    return df


def unique_wls_photons(df: pd.DataFrame) -> pd.DataFrame:
    required = set(UNIQUE_KEYS + ["ProcessName"])
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    filtered = df.loc[df["ProcessName"] == PROCESS_NAME].copy()
    return filtered.drop_duplicates(subset=UNIQUE_KEYS)


In [ ]:
csv_paths = sorted(OUTPUT_DIR.glob(CSV_GLOB))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {OUTPUT_DIR.resolve()}")

frames = [read_tools_wcsv(path) for path in csv_paths]
all_hits = pd.concat(frames, ignore_index=True)
all_unique_wls = unique_wls_photons(all_hits)

per_file_summary = []
for path, frame in zip(csv_paths, frames):
    unique_frame = unique_wls_photons(frame)
    per_file_summary.append(
        {
            "SourceFile": path.name,
            "Rows": len(frame),
            "OpWLSRows": int((frame["ProcessName"] == PROCESS_NAME).sum()),
            "UniqueOpWLSPhotons": len(unique_frame),
        }
    )

per_file_summary = pd.DataFrame(per_file_summary).sort_values("SourceFile").reset_index(drop=True)
per_file_summary

In [ ]:
combined_summary = pd.DataFrame(
    [
        {
            "TotalRows": len(all_hits),
            "TotalOpWLSRows": int((all_hits["ProcessName"] == PROCESS_NAME).sum()),
            "UniqueOpWLSPhotons": len(all_unique_wls),
        }
    ]
)
combined_summary

In [ ]:
sipm_volume_mask = all_hits["Volume"].astype(str).str.contains("sipm", case=False, na=False)
wls_sipm_hits = all_hits.loc[(all_hits["ProcessName"] == PROCESS_NAME) & sipm_volume_mask].copy()
unique_wls_seen_by_sipm = wls_sipm_hits.drop_duplicates(subset=UNIQUE_KEYS)

sipm_summary = pd.DataFrame(
    [
        {
            "UniqueOpWLSPhotonsSeenBySiPM": len(unique_wls_seen_by_sipm),
            "FractionOfUniqueOpWLSPhotons": len(unique_wls_seen_by_sipm) / len(all_unique_wls) if len(all_unique_wls) else 0.0,
        }
    ]
)
sipm_summary

In [ ]:
all_unique_wls[["SourceFile", "EventID", "TrackID", "ParentID", "OriginVolume", "Volume", "InitialEnergy"]].head(20)